In [ ]:
from your_job_offer.services.vacancies_repository.db_methods import get_all_vacancies

vacancies = get_all_vacancies()

In [3]:
from your_job_offer.services.cv_parser.parser import ResumeParser

parser = ResumeParser()
user1 = parser.parse("your_job_offer/tests/parser/files/resume1.pdf")
# user2 = parser.parse("your_job_offer/tests/parser/files/resume2.pdf")
# user3 = parser.parse("your_job_offer/tests/parser/files/resume3.pdf")

Заметки:
- в requirments есть уровень образования
- если есть какой-то уровень образования, то есть и все ниже, чтобы предложения типа среднее образование и выше работали
- если мы возьмем слишком много вакансий, то ничего страшного, если упустим - пролема

Нужные поля:
- experience
- requirment
- area

In [5]:
from enum import Enum

from your_job_offer.domain.models.jobs import Vacancy
from your_job_offer.domain.models.user import User

In [ ]:
import re
from string import punctuation

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import download as nltk_download

from pymorphy2 import MorphAnalyzer

punctuation = punctuation.replace("+", "")


class TextPreprocessor:
    def __init__(self):
        nltk_download("punkt_tab")
        nltk_download("stopwords")
        rus_stops = stopwords.words("russian")
        self.filter_token = rus_stops + [
            "знание",
            "способность",
            "умение",
            "высокий",
            "уровень",
            "степень",
            "опыт",
            "хороший",
            "практический",
            "навык",
            "качество",
            "год",
            "мидло",
            "понимание",
            "концепция",
            "некоторый",
        ]
        self.parser = MorphAnalyzer()

    @staticmethod
    def clean(word: str) -> str:
        return re.sub(r"[^A-ZА-Яa-zа-я+\s]", "", word)

    def lemmatize(self, word: str) -> str:
        return self.parser.parse(word)[0].normal_form

    def text_to_tokens(self, text: str) -> list[str]:
        text = text.lower()
        text = text.translate(
            str.maketrans(punctuation, " " * len(punctuation))
        )
        text = text.translate(str.maketrans({"\n": " ", "\t": " ", "-": " "}))
        tokenized_text = word_tokenize(text)
        clean_text = list(map(self.clean, tokenized_text))
        lemmatized_text = []
        for word in clean_text:
            word = self.lemmatize(word)
            if not (
                len(word) < 2 or word in self.filter_token
            ):  # однобуквенные слова смысла не несут
                lemmatized_text.append(word)
        return lemmatized_text

In [14]:
import numpy as np
from operator import itemgetter

def isin(skills: list[str], requirement: str | None) -> int:
    if requirement is None or len(requirement) == 0:
        return 1
    ans = 0
    for skill in skills:
        if skill in requirement:
            ans += 1
    return ans


def _match_vacancies_by_words_entry(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    [Baseline]
    проверяет, что хотя бы один скилл из user входит в хотя бы одно слово из requirement
    """
    processor = TextPreprocessor()
    user_skills = list(
        map(
            lambda skill: " ".join(processor.text_to_tokens(skill)),
            user.skills,
        )
    )
    print(user_skills)
    counts_isin = np.array(list(
        map(lambda vacancy: isin(user_skills, vacancy.requirement), vacancies)
    ))

    indexes = np.argsort(counts_isin, )[::-1][:sum(counts_isin != 0)]

    return itemgetter(*indexes)(vacancies)

In [10]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import trange


def torch_set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch_set_seed(42)
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny")


def embed_bert_cls(text):
    t = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        model_output = model(**{k: v.to(model.device) for k, v in t.items()})
    embeddings = model_output.last_hidden_state[:, 0, :]
    embeddings = torch.nn.functional.normalize(embeddings)
    return embeddings[0].cpu().numpy()


def cos_dist(x, y):
    return 1 - np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))


def _match_vacancies_by_bert(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    обрабатывает bertом, находит самые близкие вакансии(top100)
    """
    user_skill_embedding = embed_bert_cls(", ".join(user.skills))
    vacancies_skill_embeddings = np.zeros(
        (len(vacancies), len(user_skill_embedding))
    )
    for i in trange(len(vacancies)):
        vacancies_skill_embeddings[i] = embed_bert_cls(
            vacancies[i].requirement
        )
    dists = np.apply_along_axis(
        cos_dist, 1, vacancies_skill_embeddings, user_skill_embedding
    )
    indexes = np.argsort(dists)[:100]

    return itemgetter(*indexes)(vacancies)

/home/ruslan/Projects/your_job_offer2/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class MatchingEnum(Enum):
    WORDS_ENTRY = "words_entry"
    BERT = "bert"


def filter_without_skills(
    vacancies: list[Vacancy], user: User
) -> list[Vacancy]:
    """
    фильтрует то, что не смогли отфильтровать по запросам к бд, но без учёта скиллов,
    то есть поля area, experience
    """
    if user.relocation:
        return vacancies
    return vacancies  # TODO


def match_vacancies(
    vacancies: list[Vacancy],
    user: User,
    mode: MatchingEnum = MatchingEnum.WORDS_ENTRY,
) -> list[Vacancy]:
    """
    подбирает вакансии по mode, и отсортировывает их по релевантности

    :param vacancies: отфильтрованные по полям employment, schedule, buisiness_trip_readiness, relocation, min_salary, max_salary вакансии
    :param mode: способ подбора
    :return: список отсортированных вакансий
    """
    vacancies = list(filter(lambda x: not x.requirement is None, vacancies))
    vacancies = filter_without_skills(vacancies, user)
    if mode == MatchingEnum.WORDS_ENTRY:
        return _match_vacancies_by_words_entry(vacancies, user)
    if mode == MatchingEnum.BERT:
        return _match_vacancies_by_bert(vacancies, user)


def print_result(func, user, print_count: int = 20):
    vacancies_ = func(vacancies, user)
    print(f"User skills: {user.skills}")
    print(f"Найдено {len(vacancies_)} из {len(vacancies)} вакансий")
    print(f"Примеры")
    for i in range(print_count):
        print(vacancies_[i].requirement)

In [15]:
print_result(match_vacancies, user1)

[nltk_data] Downloading package punkt_tab to /home/ruslan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ruslan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['c++', 'python', 'sql', 'docker', 'git', 'bash']
User skills: ['c++', 'Python', 'SQL', 'Docker', 'git', 'bash']
Найдено 31 из 2500 вакансий
Примеры
Знание локальных сетей. Знание ОС linux. Навыки программирования(php/python/bash). Навыки работы с базами данных(mysql/sqlite). 
Опыт администрирования *nix. Знание скриптовых языков: python(будет преимуществом), bash/sh. Знание алгоритмов и структур данных(postgresql, mysql и т...
Знание linux систем. Навыки автоматизации процесса разработки в docker. Знание postgres, mysql IIS.
Администрирование Linux. Практический опыт работы в docker, gitlab.
Знание Linux (группы и права доступа, сервисы systemd, анализ производительности). - Базовое знакомство с bash. - Знание sql на уровне формирования базовых...
Server administration- Linux (Debian, Ubuntu). - work station administration (Debian, Windows, MacBook). Iptables. TCP/IP (including network troubleshooting). - bash sripts. -
Опыт работы в digital- и креативных агентствах от 5 лет на позици

In [16]:
print_result(match_vacancies, user2)

[nltk_data] Downloading package punkt_tab to /home/ruslan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ruslan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['с++', 'python', 'go', 'git', 'docker', 'django', 'mpi', 'bash', 'virtualbox']
User skills: ['С/С++', 'Python', 'Go', 'Git', 'Docker', 'Django', 'MPI', 'Bash', 'VirtualBox']
Найдено 39 из 2500 вакансий
Примеры
Cтэк: python / Go / typescript / solidity / redis / PostgreSQL / mogodb / hardhat / fastapi / asyncio / hardhat. Понимание технологических рисков DeFi, опыт аудита и написания...
Знание локальных сетей. Знание ОС linux. Навыки программирования(php/python/bash). Навыки работы с базами данных(mysql/sqlite). 
Администрирование Linux. Практический опыт работы в docker, gitlab.
Рython3+. Django 2+, опыт работы от 1 года. Django REST Framework. MySQL. MariaDB. Умение работать с git. 
Опыт администрирования *nix. Знание скриптовых языков: python(будет преимуществом), bash/sh. Знание алгоритмов и структур данных(postgresql, mysql и т...
Опыт работы в digital- и креативных агентствах от 5 лет на позиции арт-директора. Умение предлагать и обосновывать креативные решения...
Владение SQL на

In [12]:
print_result(lambda vacancies, user : match_vacancies(vacancies, user, MatchingEnum.BERT), user1)

100%|██████████| 2468/2468 [00:38<00:00, 64.32it/s] 


User skills: ['c++', 'Python', 'SQL', 'Docker', 'git', 'bash']
Найдено 100 из 2500 вакансий
Примеры
Linux(Ubuntu/CentOS). Gitlab, Nexus, HELM. Unix Shell, Python, Ansible. Apache, Tomcat, Nginx, HAProxy. MySQL, PostgreSQL.
Мы используем: GKE, K8s, PHP, Symphony, PostgreSQL, Redis, Kafka, Google object storage, React, GitLab CI/CD, Ansible, Terraform...
Java 8, Java 17, Spring boot, Java EE, JSF, REST, MariaDB, MySQL, Git.
Владение PHP => 5.2, MySQL. Владение HTML 4, CSS 2/3, javascript, jquery. Опыт работы с препроцессорами (Stylus, SASS, LESS...
Администрирование Linux, Microsoft Windows Server 2008-2022 (AD, DNS, WSUS, DHCP, LDAP, Kerberos, роли FSMO, RemoteApp, GPO, PKI. 
Windows Server, Ubuntu. Microsoft Sql Server, PostgreSql, Redis, RabbitMqAzure DevOps (aka TFS). Ansible. Graylog. Prometheus, Grafana. PowerShell. Сертификат об окончании...
Java 8-17. Maven. Hibernate. Springframework (boot, data jpa и т.д.). SQL. Docker. Git. Postgress. Kafka/rest/ и т...
Gitlab CI: настройка и 